<a href="https://colab.research.google.com/github/abraham10rod-ux/Proyecto-YOLOv8/blob/main/YOLO_Prueba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q ultralytics

import os
import glob
import subprocess
from google.colab import files
from ultralytics import YOLO
from IPython.display import HTML, display
from base64 import b64encode



¡Librerías instaladas y listas para usar!


In [16]:
from google.colab import files

print("Por favor, haz clic en el botón de abajo y selecciona tu video (ej. tu archivo .mp4):")
uploaded = files.upload()

if len(uploaded) == 0:
    print("⚠️ No se subió ningún archivo. Por favor, vuelve a ejecutar esta celda.")
else:
    video_path = list(uploaded.keys())[0]
    print(f"✅ ¡Video cargado con éxito con el nombre: {video_path}!")

Por favor, haz clic en el botón de abajo y selecciona tu video (ej. tu archivo .mp4):


Saving videoplayback.mp4 to videoplayback.mp4
✅ ¡Video cargado con éxito con el nombre: videoplayback.mp4!


In [24]:
!pip install -q ultralytics

import os
from ultralytics import YOLO

if 'video_path' in locals() and os.path.exists(video_path):
    model = YOLO('yolov8n.pt')
    print("Procesando el video....")
    results = model(source=video_path, save=True, classes=[0, 1], conf=0.3)
    print("Detección de video finalizada ")
else:
    print("ERROR:Primero debes subir un video ejecutando la Celda 2.")

Procesando el video....

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/597) /content/videoplayback.mp4: 384x640 16 persons, 8.9ms
video 1/1 (frame 2/597) /content/videoplayback.mp4: 384x640 15 persons, 9.1ms
video 1/1 (frame 3/597) /content/videoplayback.mp4: 384x640 15 persons, 9.8ms
video 1/1 (frame 4/597) /content/videoplayback.mp4: 384x640 16 persons, 9.5ms
video 1/1 (frame 5/597) /content/videoplayback.mp4: 384x640 16 persons, 9.0ms
video 1/1 (frame 6/597) /content/videoplaybac

In [25]:
import os
import glob
import subprocess
from base64 import b64encode
from IPython.display import HTML, display

predict_dirs = sorted(glob.glob('runs/detect/predict*'), key=os.path.getmtime)

if not predict_dirs:
    print(" No se encontró ninguna carpeta de predicción. Asegúrate de que la Celda 3 terminó de ejecutarse correctamente.")
else:
    latest_predict_dir = predict_dirs[-1]

    video_files = glob.glob(os.path.join(latest_predict_dir, '*.*'))

    if not video_files:
        print("No se encontró ningún video procesado dentro de la carpeta de resultados.")
    else:
        raw_output = video_files[0]
        print(f" Video procesado localizado en: {raw_output}")

        converted_output = "video_detectado_final.mp4"

        print("⚙️ Optimizando video para reproducción web...")
        cmd = f'ffmpeg -y -i "{raw_output}" -vcodec libx264 -crf 23 "{converted_output}"'
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        if os.path.exists(converted_output):
            print("✨ ¡Todo listo! Reproduciendo video:")
            mp4 = open(converted_output, 'rb').read()
            data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

            display(HTML(f'''
            <video width=650 controls autoplay loop>
                  <source src="{data_url}" type="video/mp4">
            </video>
            '''))
        else:
            print("❌ Hubo un error al convertir el video con ffmpeg.")

 Video procesado localizado en: runs/detect/predict-2/videoplayback.avi
⚙️ Optimizando video para reproducción web...
✨ ¡Todo listo! Reproduciendo video:


In [28]:
# 1. Instalar Flask y pyngrok para crear el servidor de streaming
!pip install -q flask pyngrok

from flask import Flask, Response
from pyngrok import ngrok
import cv2
import threading
import os
import glob

# ==========================================
# CONFIGURACIÓN DE NGROK (OBLIGATORIO)
# Consigue tu token gratuito en: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "TU_TOKEN_DE_NGROK_AQUI"
# ==========================================

app = Flask(__name__)

# Asegurarnos de usar el último video procesado por YOLO
predict_dirs = sorted(glob.glob('runs/detect/predict*'), key=os.path.getmtime)
if predict_dirs:
    latest_predict_dir = predict_dirs[-1]
    video_files = glob.glob(os.path.join(latest_predict_dir, '*.*'))
    if video_files:
        stream_video_path = video_files[0]
    else:
        stream_video_path = video_path
else:
    stream_video_path = video_path

def generate_frames():
    # Abrir el video procesado en bucle continuo
    while True:
        cap = cv2.VideoCapture(stream_video_path)
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            # Comprimir el fotograma a formato JPEG para la web
            ret, buffer = cv2.imencode('.jpg', frame)
            frame_bytes = buffer.tobytes()

            # Transmitir el fotograma por HTTP (Streaming MJPEG)
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n')
        cap.release()

@app.route('/video_feed')
def video_feed():
    return Response(generate_frames(),
                    mimetype='multipart/x-mixed-replace; boundary=frame')

@app.route('/')
def index():
    return '''
    <html>
        <head><title>YOLOv8 Streaming Colab</title></head>
        <body style="text-align: center; background-color: #222; color: white;">
            <h2>Transmisión en Vivo - YOLOv8 Detección de Personas</h2>
            <img src="/video_feed" width="800"/>
        </body>
    </html>
    '''

# Cerrar túneles anteriores por seguridad
ngrok.kill()

if NGROK_TOKEN == "TU_TOKEN_DE_NGROK_AQUI" or not NGROK_TOKEN.strip():
    print("⚠️  ERROR: Necesitas un token de ngrok para crear el túnel público.")
    print("1. Regístrate gratis en https://dashboard.ngrok.com")
    print("2. Copia tu 'Your Authtoken' de la plataforma.")
    print("3. Pégalo arriba en la variable NGROK_TOKEN y vuelve a ejecutar esta celda.\n")
else:
    try:
        # Configurar el token de autenticación
        ngrok.set_auth_token(NGROK_TOKEN.strip())

        # Configurar el puerto y abrir el túnel público
        port = 5000
        public_url = ngrok.connect(port)
        print(f"\n🌐 ¡Enlace de transmisión generado con éxito!")
        print(f"👉 Abre este enlace en cualquier dispositivo para ver el video: {public_url}\n")

        # Ejecutar el servidor Flask en un hilo independiente para que no bloquee Colab
        def run_flask():
            app.run(host='0.0.0.0', port=port, debug=False, use_reloader=False)

        server_thread = threading.Thread(target=run_flask)
        server_thread.daemon = True
        server_thread.start()
    except Exception as e:
        print(f"❌ Error al iniciar ngrok: {e}")

⚠️  ERROR: Necesitas un token de ngrok para crear el túnel público.
1. Regístrate gratis en https://dashboard.ngrok.com
2. Copia tu 'Your Authtoken' de la plataforma.
3. Pégalo arriba en la variable NGROK_TOKEN y vuelve a ejecutar esta celda.

